In [1]:
import os
import kagglehub
import pandas as pd

print("⏳ Ingesting raw financial data stream via Kaggle API...")
# This leverages the local cache we already created, so it will load instantly!
download_path = kagglehub.dataset_download("kartik2112/fraud-detection")
csv_path = os.path.join(download_path, "fraudTrain.csv")

# Ingest data
df = pd.read_csv(csv_path)
print(f"✅ Data successfully loaded! Shape: {df.shape[0]:,} rows and {df.shape[1]} columns.\n")

# Display the data types and look for missing fields
df.info()

⏳ Ingesting raw financial data stream via Kaggle API...
✅ Data successfully loaded! Shape: 1,296,675 rows and 23 columns.

<class 'pandas.DataFrame'>
RangeIndex: 1296675 entries, 0 to 1296674
Data columns (total 23 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   Unnamed: 0             1296675 non-null  int64  
 1   trans_date_trans_time  1296675 non-null  str    
 2   cc_num                 1296675 non-null  int64  
 3   merchant               1296675 non-null  str    
 4   category               1296675 non-null  str    
 5   amt                    1296675 non-null  float64
 6   first                  1296675 non-null  str    
 7   last                   1296675 non-null  str    
 8   gender                 1296675 non-null  str    
 9   street                 1296675 non-null  str    
 10  city                   1296675 non-null  str    
 11  state                  1296675 non-null  str    
 12  zip           

In [2]:
# Calculate basic financial statistics for transaction amounts
print("📋 TRANSACTION AMOUNT PROFILING:")
print(df['amt'].describe())

# Calculate the exact distribution of Legitimate vs Fraudulent transactions
print("\n🚨 FRAUD CLASS DISTRIBUTION:")
fraud_counts = df['is_fraud'].value_counts()
fraud_percentage = df['is_fraud'].value_counts(normalize=True) * 100

summary_df = pd.DataFrame({
    'Total Records': fraud_counts,
    'Percentage (%)': fraud_percentage
})
summary_df.index = ['Legitimate (0)', 'Fraudulent (1)']
print(summary_df)

📋 TRANSACTION AMOUNT PROFILING:
count    1.296675e+06
mean     7.035104e+01
std      1.603160e+02
min      1.000000e+00
25%      9.650000e+00
50%      4.752000e+01
75%      8.314000e+01
max      2.894890e+04
Name: amt, dtype: float64

🚨 FRAUD CLASS DISTRIBUTION:
                Total Records  Percentage (%)
Legitimate (0)        1289169       99.421135
Fraudulent (1)           7506        0.578865


In [3]:
# Filter out extreme high-value transactions
high_value_tx = df[df['amt'] > 1000]
high_value_fraud_rate = high_value_tx['is_fraud'].mean() * 100

print(f"📊 HIGH-VALUE OUTLIER ANALYSIS:")
print(f"• Number of transactions over $1,000: {len(high_value_tx):,}")
print(f"• Fraud rate among these high-value transactions: {high_value_fraud_rate:.2f}%")

📊 HIGH-VALUE OUTLIER ANALYSIS:
• Number of transactions over $1,000: 3,936
• Fraud rate among these high-value transactions: 24.11%


In [4]:
# Cell 4: Data Transformation and Cleaning
print("🧼 Beginning targeted data cleaning based on analyst insights...")

# 1. Drop useless index noise columns
columns_before = df.shape[1]
df_cleaned = df.drop(columns=['Unnamed: 0'], errors='ignore')
print(f"• Removed structural noise columns. Column count shifted from {columns_before} to {df_cleaned.shape[1]}.")

# 2. Convert date/time string into an active database datetime format
df_cleaned['trans_date_trans_time'] = pd.to_datetime(df_cleaned['trans_date_trans_time'])
print("• Converted 'trans_date_trans_time' from string objects to high-speed datetime format.")

# 3. Add a business logic flag: High Value Risk Alert
df_cleaned['high_value_risk'] = df_cleaned['amt'].apply(lambda x: 1 if x > 1000 else 0)
print("• Feature Engineered 'high_value_risk' binary column for transactions over $1,000.")

print("\n🎯 PREVIEW OF TRANSFORMED STREAM STRUCTURE:")
print(df_cleaned[['trans_date_trans_time', 'merchant', 'amt', 'high_value_risk', 'is_fraud']].head(5))

🧼 Beginning targeted data cleaning based on analyst insights...
• Removed structural noise columns. Column count shifted from 23 to 22.
• Converted 'trans_date_trans_time' from string objects to high-speed datetime format.
• Feature Engineered 'high_value_risk' binary column for transactions over $1,000.

🎯 PREVIEW OF TRANSFORMED STREAM STRUCTURE:
  trans_date_trans_time                            merchant     amt  \
0   2019-01-01 00:00:18          fraud_Rippin, Kub and Mann    4.97   
1   2019-01-01 00:00:44     fraud_Heller, Gutmann and Zieme  107.23   
2   2019-01-01 00:00:51                fraud_Lind-Buckridge  220.11   
3   2019-01-01 00:01:16  fraud_Kutch, Hermiston and Farrell   45.00   
4   2019-01-01 00:03:06                 fraud_Keeling-Crist   41.96   

   high_value_risk  is_fraud  
0                0         0  
1                0         0  
2                0         0  
3                0         0  
4                0         0  
